## Notebook Workflow Structure

This notebook systematically tests Epic Encounters-related features extraction and retrieval functionality:

---
**Cells/Workflow Order:**

| Cell # | Purpose | Expected Results |
|--------|---------|------------------|
| 1-2 | Environment setup (imports, paths) | Python path configured correctly |
| 3 | Cleanup previous test outputs | No stale data remains |
| 4 | Start Elasticsearch container | Container running on port 9200 |
| 5 | Create credentials file | `test_elastic_credentials.py` generated |
| 6-7 | Populate dummy patient data + epic encounters data | 5 patients with encounters in Elasticsearch cluster |
| 8 | Index refresh verification | All indices have documents |
| 9-10 | Initialize database and logging | Database at `outputs/temp_epic_encounters_db.sqlite` |
| 11-12 | Create pat2vec config with epic_encounters mode | Config object created successfully |
| 13-14 | Run pat2vec pipeline | Pipeline processes patients without errors |
| 15-16 | Extract all features from database | Features DataFrame populated |
| 17 | Output features dataframe preview | Data extracted displayed |
| 18 | Epic encounters data retrieval test | `get_epic_encounters()` returns patient features, non-empty |
| 19-20 | Feature merge functionality | `merge_epic_encounters_data()` creates CSV file with data |
| 21-22 | Database and project cleanup | All temporary files deleted |
| 23 | Final verification | All assertions pass |

---
**Test Failure Conditions:**
- Any cell raises unhandled exception
- Elasticsearch container fails to start
- Empty patient list after population  
- Empty DataFrame returned from feature extraction
- Epic encounters data retrieval returns empty or None result
- Patient count does not match expected (5 patients)
- Merge function produces empty result - FATAL ERROR
- Cleanup verification fails (residual files remain)

In [ ]:
import os
import random
import shutil
import sys

import numpy as np

In [ ]:
random_seed_value = 42

np.random.seed(random_seed_value)
random.seed(random_seed_value)

In [ ]:
current_dir = os.getcwd()
grandparent_dir = os.path.dirname(os.path.dirname(current_dir))

sys.path.insert(0, os.path.join(grandparent_dir, "pat2vec"))
sys.path.append(grandparent_dir)
pat2vec_dir = os.path.abspath(os.path.join(grandparent_dir, "pat2vec"))

print(f"Pat2vec path: {pat2vec_dir}")

In [ ]:
for dir_to_remove in ["epic_encounters_test_project"]:
    try:
        shutil.rmtree(dir_to_remove, ignore_errors=True)
    except Exception as e:
        raise RuntimeError(
            f"Failed to clean up '{dir_to_remove}' directory: {e}. "
            "Critical error - cannot start with stale data."
        ) from e

print("Previous outputs cleaned.")

In [ ]:
from pat2vec.util.docker_elastic import ElasticContainer

es_container = ElasticContainer()
es_container.stop()

print("Starting Elasticsearch container (this may take a few seconds)...")
if not es_container.start():
    raise RuntimeError(
        "Failed to start Elasticsearch container. Check if Docker is running."
    )

host, username, password = es_container.get_credentials()

creds_filename = "test_elastic_credentials.py"
creds_content = f"""
username = "{username}"
password = "{password}"
api_key = None
hosts = ["{host}"]
"""

with open(creds_filename, "w") as f:
    f.write(creds_content)

print(f"Created '{creds_filename}' pointing to test cluster at {host}")

In [ ]:
from pat2vec.util.config_pat2vec import config_class

schema_path = os.path.abspath("test_files/elastic_schemas.json")
config_populate = config_class(
    proj_name="epic_encounters_test_project",
    credentials_path=creds_filename,
    test_schema_path=schema_path,
    testing=True,
    testing_elastic=True,
    global_start_year=2020,
    global_start_month=1,
    global_start_day=1,
    global_end_year=2023,
    global_end_month=12,
    global_end_day=31,
)

In [ ]:
from pat2vec.util.get_dummy_data_cohort_searcher import populate_elastic_with_dummy_data

print("Populating test Elasticsearch cluster with dummy data...")
patient_ids = populate_elastic_with_dummy_data(config_populate, n_patients=5)

print()
print("Population complete.")
print(f"Generated {len(patient_ids)} dummy patients.")
print(f"Patient IDs: {patient_ids}")

In [ ]:
from pat2vec.pat2vec_search.cogstack_search_methods import initialize_cogstack_client

cs = initialize_cogstack_client(config_populate)

indices = ["epr_documents", "basic_observations", "observations", "order", "pims_apps"]
print("Refreshing indices...")
cs.elastic.indices.refresh(index=indices, ignore_unavailable=True)
print("Indices refreshed.")

print()
print("Index Status:")
for index in indices:
    try:
        if cs.elastic.indices.exists(index=index):
            count = cs.elastic.count(index=index)["count"]
            print(f"  - {index:<20}: {count} documents")
        else:
            raise RuntimeError(f"Index not created: {index}")
    except Exception as e:
        raise RuntimeError(f"Error checking index {index}: {e}")

In [ ]:
from pat2vec.util.elasticsearch_methods import ingest_data_to_elasticsearch
from pat2vec.util.get_dummy_data_cohort_searcher import generate_epic_encounters_data
import pandas as pd

encounters_dfs = []
for pid in patient_ids:
    df = generate_epic_encounters_data(
        num_rows=2,
        entered_list=[pid],
        global_start_year=int(config_populate.global_start_year),
        global_start_month=int(config_populate.global_start_month),
        global_end_year=int(config_populate.global_end_year),
        global_end_month=int(config_populate.global_end_month),
    )
    encounters_dfs.append(df)

df_encounters = (
    pd.concat(encounters_dfs, ignore_index=True)
    if len(encounters_dfs) > 1
    else encounters_dfs[0]
)
df_encounters = df_encounters.where(pd.notnull(df_encounters), None)

ingest_data_to_elasticsearch(df_encounters, "epic_encounters", es_client=cs.elastic)
cs.elastic.indices.refresh(index="epic_encounters")

print(
    f"Ingested {len(df_encounters)} epic_encounters records for {len(patient_ids)} patients"
)

In [ ]:
PROJ_NAME = "epic_encounters_test_project"
DB_FILENAME = "temp_epic_encounters_db.sqlite"
DB_PATH = os.path.join(PROJ_NAME, "outputs", DB_FILENAME)

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

db_connection_string = f"sqlite:///{DB_PATH}"
print(f"Database connection string set to: {db_connection_string}")

In [ ]:
from pat2vec.util.logger_setup import setup_logger

logger = setup_logger()
print("Logger initialized.")

In [ ]:
from pat2vec.util.config_pat2vec import config_class

config_obj = config_class(
    proj_name=PROJ_NAME,
    credentials_path=creds_filename,
    current_path_dir="",
    main_options={"epic_encounters": True},
    batch_mode=True,
    verbosity=0,
    random_seed_val=random_seed_value,
    testing=True,
    testing_elastic=True,
    dummy_medcat_model=True,
    use_controls=False,
    medcat=False,
    start_time=None,
    patient_id_column_name="client_idcode",
    annot_filter_options={},
    shuffle_pat_list=False,
    storage_backend="database",
    db_connection_string=db_connection_string,
    all_patient_list=patient_ids,
)

print(
    "pat2vec configuration created with epic_encounters-only sources and database backend."
)

In [ ]:
from pat2vec.main_pat2vec import main

try:
    pat2vec_obj = main(
        cogstack=True,
        use_filter=False,
        json_filter_path=None,
        random_seed_val=random_seed_value,
        hostname=None,
        config_obj=config_obj,
    )
except FileNotFoundError as e:
    raise RuntimeError(
        f"Failed to initialize pipeline: config path invalid. Error details: {e}."
    ) from e
except ValueError as e:
    raise RuntimeError(
        f"Failed to initialize pipeline: invalid configuration. Error details: {e}."
    ) from e
except RuntimeError as e:
    raise RuntimeError(
        f"Failed to initialize pipeline: initialization failure. Error details: {e}."
    ) from e
except Exception as e:
    raise RuntimeError(
        f"Failed to initialize pipeline: unexpected error. Error details: {e}."
    ) from e

print("pat2vec object initialized.")
print(f"Patient list: {pat2vec_obj.all_patient_list}")

In [ ]:
if not pat2vec_obj.all_patient_list:
    raise RuntimeError(
        "No patients in patient list after initialization. "
        "This indicates a critical failure in data loading or filtering."
    )

print(f"Processing patient: {pat2vec_obj.all_patient_list[0]}")

try:
    pat2vec_obj.pat_maker(0)
except Exception as e:
    raise RuntimeError(
        f"Failed to process patient 0 with pat_maker: {e}. "
        "Critical error - pipeline failed to extract features."
    ) from e

print("Patient feature extraction complete.")

In [ ]:
from pat2vec.util.helper_functions import get_all_features

all_features = get_all_features(config_obj)

if all_features.empty:
    raise RuntimeError(
        "FATAL ERROR: get_all_features returned an empty DataFrame. "
        "This indicates a critical failure in the pat2vec pipeline. "
        "No features were extracted or saved to the database."
    )

print(f"Successfully retrieved {len(all_features)} rows from database.")

In [ ]:
all_features_alt = pat2vec_obj.get_all_features()

if all_features_alt.empty:
    raise RuntimeError(
        "FATAL ERROR: pat2vec_obj.get_all_features() returned an empty DataFrame. "
        "This indicates a critical failure in feature storage."
    )

print(f"pat2vec_obj.get_all_features(): {len(all_features_alt)} rows retrieved.")

In [ ]:
print()
print("=== OUTPUT FEATURES DATAFRAME ===")
print(f"Shape: {all_features.shape}")
print(f"Total features: {len(all_features.columns)}")

if not all_features.empty:
    print()
    print("First 3 rows:")
    print(all_features.head(3))
else:
    raise RuntimeError(
        "DataFrame is empty after feature extraction. Critical error - no features to extract."
    )

In [ ]:
print("\n=== DEMONSTRATING DATA RETRIEVAL FOR EPIC ENCOUNTERS MODE ===")

all_pat_list = pat2vec_obj.all_patient_list

from pat2vec.pat2vec_get_methods.get_method_epic_encounters import get_epic_encounters

pat_batch = pd.DataFrame()

encounter_data = get_epic_encounters(
    current_pat_client_id_code=all_pat_list[0],
    target_date_range=(2020, 1, 1, 2023, 12, 31),
    pat_batch=pat_batch,
    config_obj=config_obj,
)

if encounter_data is None:
    raise RuntimeError(
        "FATAL ERROR: get_epic_encounters returned empty result. "
        "This indicates a critical failure in epic encounters feature extraction."
    )

if isinstance(encounter_data, list) and len(encounter_data) == 0:
    raise RuntimeError(
        "FATAL ERROR: get_epic_encounters returned empty list. "
        "This indicates a critical failure in epic encounters feature extraction."
    )

if isinstance(encounter_data, list):
    patient_count = len(encounter_data)
else:
    patient_count = 1

print(f"Retrieved epic encounters data for {patient_count} patient(s)")
if isinstance(encounter_data, list):
    print(f"\nEpic Encounters columns: {list(encounter_data[0].columns)}")
    print("\nSample epic encounters features:")
    print(encounter_data[0])
else:
    print(f"\nEpic Encounters columns: {list(encounter_data.columns)}")
    print("\nSample epic encounters features:")
    print(encounter_data)

In [ ]:
print("\n=== DEMONSTRATING FEATURE MERGE FUNCTIONALITY ===")

merged_df = merge_epic_encounters_data(config_obj)

if merged_df is None or (isinstance(merged_df, list) and len(merged_df) == 0):
    raise RuntimeError(
        "FATAL ERROR: merge_epic_encounters_data returned empty result. "
        "This indicates a critical failure in epic encounters feature extraction."
    )

if isinstance(merged_df, list) and len(merged_df) == 0:
    raise RuntimeError(
        "FATAL ERROR: merge_epic_encounters_data returned empty list. "
        "This indicates a critical failure in epic encounters feature extraction."
    )

print(f"Merged epic_encounters data shape: {merged_df.shape}")
if merged_df.empty:
    raise RuntimeError(
        "FATAL ERROR: Merged epic_encounters dataframe is empty. "
        "This indicates the pat2vec pipeline did not save data to database "
        "or no epic encounters were found for the patients."
    )
print(f"\nColumns: {list(merged_df.columns)}")
print("\nData preview:")
print(merged_df.head())

In [ ]:
print("\n=== DATABASE AND PROJECT CLEANUP ===")

try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
        print(f"Removed database: {DB_PATH}")
except Exception as e:
    raise RuntimeError(
        f"Failed to remove database file '{DB_PATH}': {e}. Critical error - cleanup incomplete."
    ) from e

try:
    if os.path.exists(PROJ_NAME):
        shutil.rmtree(PROJ_NAME, ignore_errors=False)
        print(f"Removed project directory: {PROJ_NAME}")
except Exception as e:
    raise RuntimeError(
        f"Failed to remove '{PROJ_NAME}' directory: {e}. Critical error - cleanup incomplete."
    ) from e

try:
    if os.path.exists(creds_filename):
        os.remove(creds_filename)
        print(f"Removed Elasticsearch credentials: {creds_filename}")
except Exception as e:
    raise RuntimeError(
        f"Failed to remove Elasticsearch credentials file '{creds_filename}': {e}. "
        "Critical error - cleanup incomplete."
    ) from e

In [ ]:
print("\n=== FINAL VERIFICATION ===")

assert not os.path.exists(DB_PATH), "Database file still exists!"
assert not os.path.exists(PROJ_NAME), "Project directory still exists!"
assert not os.path.exists(
    creds_filename
), "Elasticsearch credentials file still exists!"

print("All cleanup verified - no residual files remain.")
print("\n=== TEST SUCCESSFUL ===")